In [ ]:
!pip install qiskit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 5.0 MB/s eta 0:00:00


# Exercise 1.1

In [ ]:
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager

# 1. Circuit
qc = QuantumCircuit(3)
qc.h(0)
qc.h(0)
qc.ccx(0, 1, 2)
qc.cx(1, 2)
qc.cx(1, 2)

print("CIRCUIT BEFORE INIT STAGE")
print(qc.draw(output="text"))
print("\nGate count:")
print(qc.count_ops())
print("Depth:", qc.depth())

pm = generate_preset_pass_manager(
    optimization_level=0,
    basis_gates=["h", "cx", "t", "tdg"],
    coupling_map=[[0, 1], [1, 2]]
)
init_circuit = pm.init.run(qc)

print("CIRCUIT AFTER INIT STAGE - OPTIMIZATION LEVEL = 0")
print(init_circuit.draw(output="text"))
print("\nGate count:")
print(init_circuit.count_ops())
print("Depth:", init_circuit.depth())

pm = generate_preset_pass_manager(
    optimization_level=1,
    basis_gates=["h", "cx", "t", "tdg"],
    coupling_map=[[0, 1], [1, 2]]
)
init_circuit = pm.init.run(qc)

print("CIRCUIT AFTER INIT STAGE - OPTIMIZATION LEVEL = 1")
print(init_circuit.draw(output="text"))
print("\nGate count:")
print(init_circuit.count_ops())
print("Depth:", init_circuit.depth())

CIRCUIT BEFORE INIT STAGE
     ┌───┐┌───┐               
q_0: ┤ H ├┤ H ├──■────────────
     └───┘└───┘  │            
q_1: ────────────■────■────■──
               ┌─┴─┐┌─┴─┐┌─┴─┐
q_2: ──────────┤ X ├┤ X ├┤ X ├
               └───┘└───┘└───┘

Gate count:
OrderedDict({'h': 2, 'cx': 2, 'ccx': 1})
Depth: 5
CIRCUIT AFTER INIT STAGE - OPTIMIZATION LEVEL = 0
     ┌───┐┌───┐                                        ┌───┐                
q_0: ┤ H ├┤ H ├─────────■─────────────────────■────■───┤ T ├───■────────────
     └───┘└───┘         │             ┌───┐   │  ┌─┴─┐┌┴───┴┐┌─┴─┐          
q_1: ───────■───────────┼─────────■───┤ T ├───┼──┤ X ├┤ Tdg ├┤ X ├──■────■──
     ┌───┐┌─┴─┐┌─────┐┌─┴─┐┌───┐┌─┴─┐┌┴───┴┐┌─┴─┐├───┤└┬───┬┘└───┘┌─┴─┐┌─┴─┐
q_2: ┤ H ├┤ X ├┤ Tdg ├┤ X ├┤ T ├┤ X ├┤ Tdg ├┤ X ├┤ T ├─┤ H ├──────┤ X ├┤ X ├
     └───┘└───┘└─────┘└───┘└───┘└───┘└─────┘└───┘└───┘ └───┘      └───┘└───┘

Gate count:
OrderedDict({'cx': 8, 'h': 4, 't': 4, 'tdg': 3})
Depth: 13
CIRCUIT AFTER INIT STAGE - OPTIMI

# Exercise 1.2

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, ClassicalRegister
from qiskit.circuit.library import (
    HGate, XGate, ZGate, TGate, TdgGate, SGate, SdgGate,
    RZGate, RXGate, CXGate, CZGate,
)
from qiskit.quantum_info import Operator, Statevector
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import (
    RemoveIdentityEquivalent, RemoveDiagonalGatesBeforeMeasure,
    InverseCancellation, CommutativeInverseCancellation,
    Collect2qBlocks, ConsolidateBlocks, Split2QUnitaries,
)


# ---------------------------------------------------------------------------
# 1. Random circuit: n random qubits, n*n random gates
# ---------------------------------------------------------------------------
def random_nn(pool, seed=None, n_range=(3, 6), pair_prob=0.3, gaps=(0, 1), measure=True):
    rng = np.random.default_rng(seed)
    n = int(rng.integers(n_range[0], n_range[1] + 1))

    def rand_op():
        k, make = pool[rng.integers(len(pool))]
        return make(rng), rng.choice(n, k, replace=False).tolist()

    ops = []
    while len(ops) < n * n:
        g, q = rand_op()
        ops.append((g, q))
        if rng.random() < pair_prob:
            ops += [rand_op() for _ in range(rng.choice(gaps))]
            ops.append((g.inverse(), q))

    qc = QuantumCircuit(n)
    for g, q in ops[: n * n]:
        qc.append(g, q)
    if measure:
        qc.add_register(ClassicalRegister(n))
        qc.measure(range(n), range(n))
    return qc


# ---------------------------------------------------------------------------
# 2. Equivalence check and pass runner
# ---------------------------------------------------------------------------
def verify(a, b, prob_only=False):
    a = a.remove_final_measurements(inplace=False)
    b = b.remove_final_measurements(inplace=False)
    if prob_only:
        return bool(np.allclose(Statevector(a).probabilities(),
                                Statevector(b).probabilities()))
    return Operator(a).equiv(Operator(b))

results = []


def run_pass(qc, passes, name, prob_only=False):
    out = PassManager(passes).run(qc)
    ok = verify(qc, out, prob_only)
    print(f"{'=' * 20} {name} {'=' * 20}")
    print("BEFORE")
    print(qc.draw(fold=-1))
    print("AFTER")
    print(out.draw(fold=-1))
    print("ops before:", dict(qc.count_ops()))
    print("ops after :", dict(out.count_ops()))
    print(f"size {qc.size()} -> {out.size()} | depth {qc.depth()} -> {out.depth()}"
          f" | equivalent: {ok}\n")
    results.append((name, qc.size(), out.size(), qc.depth(), out.depth(), ok))
    return out


# ---------------------------------------------------------------------------
# 3. Generate one circuit
# ---------------------------------------------------------------------------
pool = [
    (1, lambda r: HGate()), (1, lambda r: XGate()), (1, lambda r: ZGate()),
    (1, lambda r: TGate()), (1, lambda r: SGate()),
    (1, lambda r: RZGate(r.choice([0, 2 * np.pi, np.pi / 4]))),
    (1, lambda r: RXGate(r.choice([0, 2 * np.pi, np.pi / 3]))),
    (2, lambda r: CXGate()), (2, lambda r: CZGate()),
]

seed = 7
qc = random_nn(pool, seed)
print(f"n = {qc.num_qubits}, gates = {qc.num_qubits ** 2}")
print(qc.draw(fold=-1))
print()

# ---------------------------------------------------------------------------
# 4. Run the same circuit through each pass
# ---------------------------------------------------------------------------
inv = InverseCancellation([HGate(), XGate(), ZGate(), CXGate(), CZGate(),
                           (TGate(), TdgGate()), (SGate(), SdgGate())])

run_pass(qc, RemoveIdentityEquivalent(), "RemoveIdentityEquivalent")
run_pass(qc, RemoveDiagonalGatesBeforeMeasure(), "RemoveDiagonalGatesBeforeMeasure",
         prob_only=True)
run_pass(qc, inv, "InverseCancellation")
run_pass(qc, CommutativeInverseCancellation(), "CommutativeInverseCancellation")

cons = run_pass(qc, [Collect2qBlocks(),
                     ConsolidateBlocks(basis_gates=["cx", "u"], force_consolidate=True)],
                "ConsolidateBlocks")
split = run_pass(cons, Split2QUnitaries(), "Split2QUnitaries")
print("Split2QUnitaries vs original circuit equivalent:", verify(qc, split), "\n")

# ---------------------------------------------------------------------------
# 5. Summary table
# ---------------------------------------------------------------------------
print(f"{'Pass':<36}{'Size':>12}{'Depth':>12}  Equiv")
for name, s0, s1, d0, d1, ok in results:
    print(f"{name:<36}{f'{s0}->{s1}':>12}{f'{d0}->{d1}':>12}  {ok}")

n = 6, gates = 36
                                                 ┌───┐┌────────┐┌─────────┐                                                          ┌─┐   
 q_0: ─────■───────■────────────────■─────────■──┤ H ├┤ Rx(2π) ├┤ Rx(-2π) ├────────────────────────────────────────────────────────■─┤M├───
         ┌─┴─┐     │  ┌───┐┌─────┐  │  ┌───┐  │  ├───┤└─┬───┬──┘└─────────┘                        ┌─┐                             │ └╥┘   
 q_1: ───┤ X ├─────┼──┤ T ├┤ Tdg ├──┼──┤ X ├──┼──┤ X ├──┤ S ├─────────────────────────■────────────┤M├─────────────────────────────┼──╫────
         └───┘     │  ├───┤└┬───┬┘  │  └─┬─┘  │  └─┬─┘┌─┴───┴─┐  ┌───────┐ ┌────────┐ │ ┌─────────┐└╥┘     ┌─┐                     │  ║    
 q_2: ─────────────■──┤ X ├─┤ S ├───┼────┼────┼────┼──┤ Rz(0) ├──┤ Rz(0) ├─┤ Rx(2π) ├─┼─┤ Rz(π/4) ├─╫──────┤M├─────────────────────┼──╫────
         ┌───┐        └───┘ └───┘ ┌─┴─┐  │  ┌─┴─┐  │  └─┬───┬─┘  └─┬───┬─┘ └─┬───┬──┘ │ └┬────────┤ ║ ┌───┐└╥┘┌───┐┌───┐┌────────┐ │  ║ ┌─┐
 q

# Exercise 1.3

In [ ]:
from qiskit.synthesis import (
    OneQubitEulerDecomposer, TwoQubitBasisDecomposer, TwoQubitWeylDecomposition,
)
from qiskit.transpiler.passes import UnitarySynthesis

print("Input (Split2QUnitaries output):", dict(split.count_ops()), "\n")

print(split.draw(fold=-1))

euler = OneQubitEulerDecomposer(basis="ZYZ")
kak = TwoQubitBasisDecomposer(CXGate(), euler_basis="ZYZ")

# ---- 3.1 Euler decomposition of every 1-qubit unitary --------------------
bases = ["ZYZ", "ZXZ", "U3", "ZSX"]
k = 0
for inst in split.data:
    if inst.operation.name == "unitary" and inst.operation.num_qubits == 1:
        U = inst.operation.to_matrix()
        th, ph, la, ga = euler.angles_and_phase(U)
        counts = {b: OneQubitEulerDecomposer(b)(U).size() for b in bases}
        q = split.find_bit(inst.qubits[0]).index
        k += 1

# ---- 3.2 KAK decomposition of every 2-qubit unitary ----------------------
k = 0
for inst in split.data:
    if inst.operation.name == "unitary" and inst.operation.num_qubits == 2:
        U = inst.operation.to_matrix()
        w = TwoQubitWeylDecomposition(U)
        qs = tuple(split.find_bit(q).index for q in inst.qubits)
        k += 1

# ---- 3.3 Replace every unitary by its decomposition ----------------------
dec = QuantumCircuit(*split.qregs, *split.cregs)
for inst in split.data:
    op = inst.operation
    if op.name == "unitary" and op.num_qubits == 1:
        dec.compose(euler(op.to_matrix()), inst.qubits, inplace=True)
    elif op.name == "unitary" and op.num_qubits == 2:
        dec.compose(kak(op.to_matrix()), inst.qubits, inplace=True)
    else:
        dec.append(inst)

print(dec.draw(fold=-1))
print("ops:", dict(dec.count_ops()))
print(f"size {split.size()} -> {dec.size()} | depth {split.depth()} -> {dec.depth()}")
print("equivalent to Split2Q output:", verify(split, dec))
print("equivalent to original circuit:", verify(qc, dec))
print()

# ---- 3.4 Compare with Qiskit's UnitarySynthesis pass ---------------------
syn = PassManager(UnitarySynthesis(basis_gates=["cx", "rz", "ry"])).run(split)
print("ops:", dict(syn.count_ops()))
print(f"size {syn.size()} | depth {syn.depth()} | equivalent: {verify(qc, syn)}")
print(f"CX: manual {dec.count_ops().get('cx', 0)} vs UnitarySynthesis {syn.count_ops().get('cx', 0)}")

Input (Split2QUnitaries output): {'unitary': 9, 'measure': 6} 

global phase: 7π/4
      ┌──────────┐            ┌──────────┐┌─────────┐    ┌──────────┐   ┌─┐   
 q_0: ┤0         ├────────────┤0         ├┤ Unitary ├────┤1         ├───┤M├───
      │  Unitary │┌─────────┐ │          │├─────────┴┐   │          │┌─┐└╥┘   
 q_1: ┤1         ├┤ Unitary ├─┤  Unitary ├┤0         ├───┤          ├┤M├─╫────
      └──────────┘└─────────┘ │          ││          │┌─┐│  Unitary │└╥┘ ║    
 q_2: ────────────────────────┤1         ├┤          ├┤M├┤          ├─╫──╫────
      ┌─────────┐ ┌──────────┐└──────────┘│  Unitary │└╥┘│          │ ║  ║ ┌─┐
 q_3: ┤ Unitary ├─┤0         ├────────────┤          ├─╫─┤0         ├─╫──╫─┤M├
      └─────────┘ │          │            │          │ ║ └───┬─┬────┘ ║  ║ └╥┘
 q_4: ────────────┤  Unitary ├────────────┤1         ├─╫─────┤M├──────╫──╫──╫─
      ┌─────────┐ │          │    ┌─┐     └──────────┘ ║     └╥┘      ║  ║  ║ 
 q_5: ┤ Unitary ├─┤1         ├────┤M├───────────

# Exercise 2.1

In [ ]:
import math
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator
from qiskit.transpiler import CouplingMap, PassManager
from qiskit.transpiler.passes import (
    VF2Layout, SabreLayout, TrivialLayout, SetLayout, SabreSwap,
    FullAncillaAllocation, EnlargeWithAncilla, ApplyLayout, CheckMap,
)

def random_layout_circuit(seed=None, n_range=(4, 7), p2q=0.4):
    rng = np.random.default_rng(seed)
    n = int(rng.integers(n_range[0], n_range[1] + 1))
    tree = [(int(rng.integers(i)), i) for i in range(1, n)]
    qc = QuantumCircuit(n)
    for _ in range(n * n):
        if rng.random() < p2q:
            a, b = tree[rng.integers(len(tree))]
            qc.cx(*((a, b) if rng.random() < 0.5 else (b, a)))
        else:
            q = int(rng.integers(n))
            [qc.h, qc.sx, lambda q: qc.rz(rng.uniform(0, 2 * np.pi), q)][rng.integers(3)](q)
    return qc

def place(cm):
    return [FullAncillaAllocation(cm), EnlargeWithAncilla(), ApplyLayout()]

def is_mapped(out, cm):
    pm = PassManager(CheckMap(cm)); pm.run(out)
    return pm.property_set["is_swap_mapped"]

def equiv(orig, out):                       # pad ancillas, undo initial/final layout
    ref = QuantumCircuit(out.num_qubits).compose(orig, range(orig.num_qubits))
    return Operator.from_circuit(out).equiv(Operator(ref))

def v2p(layout, qc):                        # {virtual: physical}
    return {qc.find_bit(v).index: p for v, p in layout.get_virtual_bits().items() if v in qc.qubits}

seed = 2
qc = random_layout_circuit(seed)
n = qc.num_qubits
maps = {"line": CouplingMap.from_line(n),
        "ring": CouplingMap.from_ring(n),
        f"grid 2x{math.ceil(n / 2)}": CouplingMap.from_grid(2, math.ceil(n / 2))}
print(f"n = {n}, gates = {qc.size()}")
print(qc.draw(fold=-1))

edges = sorted({tuple(sorted(qc.find_bit(q).index for q in i.qubits))
                for i in qc.data if i.operation.num_qubits == 2})
deg = np.bincount(np.array(edges).ravel(), minlength=n)
print("\n==== 2.1 VF2Layout ====")
print("Interaction graph edges:", edges, "| max degree:", deg.max())
for name, cm in maps.items():
    pm = PassManager(VF2Layout(cm, seed=seed)); pm.run(qc)
    lay = pm.property_set["layout"]
    cm_deg = max(cm.graph.out_degree(i) for i in cm.physical_qubits)
    print(f"\n[{name}] max degree {cm_deg} | {pm.property_set['VF2Layout_stop_reason'].name}")
    if lay is None:
        continue
    out = PassManager([SetLayout(lay)] + place(cm)).run(qc)
    print("layout v->p:", v2p(lay, qc))
    print(out.draw(fold=-1))
    print(f"no SWAP needed: {is_mapped(out, cm)} | equivalent: {equiv(qc, out)}")

n = 7, gates = 49
     ┌───┐          ┌───┐                                                         ┌────┐┌────────────┐┌───┐          ┌───┐          ┌───┐          ┌───┐              ┌───┐┌────────────┐                    ┌───┐
q_0: ┤ X ├──────────┤ X ├────────────────■────────────────────────────────■───────┤ √X ├┤ Rz(2.5585) ├┤ X ├──■───────┤ H ├──────────┤ X ├──■────■──┤ X ├──────────────┤ X ├┤ Rz(5.6758) ├────────────────────┤ X ├
     └─┬─┘          └─┬─┘              ┌─┴─┐     ┌───────────┐┌────┐      │       ├───┬┘└────────────┘└─┬─┘  │  ┌────┴───┴────┐┌───┐└─┬─┘┌─┴─┐  │  └─┬─┘┌────────────┐└─┬─┘└────────────┘     ┌─────────────┐└─┬─┘
q_1: ──┼────■─────────┼─────────■──────┤ X ├─────┤ Rz(4.873) ├┤ √X ├──────┼───────┤ H ├─────────────────■────┼──┤ Rz(0.26397) ├┤ X ├──┼──┤ X ├──┼────■──┤ Rz(3.8537) ├──┼──────────────────■──┤ Rz(0.33133) ├──■──
       │    │         │         │  ┌───┴───┴────┐└───────────┘├───┬┘    ┌─┴─┐     └───┘                    ┌─┴─┐└────┬────┬───┘└─┬─┘  │  └

# Exercise 2.2

In [ ]:
print("\n==== 2.2 SabreLayout ====")
print(f"{'map':<10}{'method':<20}{'swap':>5}{'cx':>5}{'depth':>7}  mapped  equiv")
for name, cm in maps.items():
    runs = {"Trivial+SabreSwap": [TrivialLayout(cm)] + place(cm) + [SabreSwap(cm, seed=seed)],
            "SabreLayout": [SabreLayout(cm, seed=seed)]}
    for meth, passes in runs.items():
        out = PassManager(passes).run(qc)
        ops = out.count_ops()
        print(f"{name:<10}{meth:<20}{ops.get('swap', 0):>5}{ops.get('cx', 0):>5}{out.depth():>7}"
              f"  {str(is_mapped(out, cm)):<7} {equiv(qc, out)}")

sab = PassManager(SabreLayout(maps["line"], seed=seed)).run(qc)
print("\nSabreLayout on line:")
print(sab.draw(fold=-1))
print("initial layout v->p:", v2p(sab.layout.initial_layout, qc))
print("final layout   v->p:", dict(enumerate(sab.layout.final_index_layout())))


==== 2.2 SabreLayout ====
map       method               swap   cx  depth  mapped  equiv
line      Trivial+SabreSwap      19   23     38  True    True
line      SabreLayout             5   23     23  True    True
ring      Trivial+SabreSwap      12   23     30  True    True
ring      SabreLayout             4   23     23  True    True
grid 2x4  Trivial+SabreSwap       6   23     27  True    True
grid 2x4  SabreLayout             0   23     18  True    True

SabreLayout on line:
         ┌────┐    ┌───┐                                                                                                                                                                                           
q_5 -> 0 ┤ √X ├────┤ H ├──────────────────────────────────────────────────────X─────────────────────────X──────────────────────────────────────────────────────────────────────────────────────────────────────────
         └────┘┌───┴───┴────┐    ┌───┐     ┌────┐┌────────────┐    ┌────┐     │        ┌───┐

# Exercise 3.1

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import CXGate
from qiskit.transpiler import Target, InstructionProperties, PassManager
from qiskit.transpiler.passes import VF2Layout, SabreLayout, VF2PostLayout, ApplyLayout

# ---------------- Chip: 0 -5%- 1 -4%- 2 -1%- 3 -1%- 4 -3%- 5 --------------------
err = {(0, 1): 0.05, (1, 2): 0.04, (2, 3): 0.01, (3, 4): 0.01, (4, 5): 0.03}
target = Target(num_qubits=6)
target.add_instruction(CXGate(), {e: InstructionProperties(error=p)
                                  for (a, b), p in err.items() for e in [(a, b), (b, a)]})
cm = target.build_coupling_map()

# ---------------- Circuit: cx(0,1), cx(1,2), cx(0,2) -----------------------------
qc = QuantumCircuit(3)
qc.cx(0, 1); qc.cx(1, 2); qc.cx(0, 2)
print(qc.draw())

def fidelity(c):                         # F = prod (1 - err)^n, SWAP = 3 CX
    F = 1.0
    for i in c.data:
        e = tuple(sorted(c.find_bit(q).index for q in i.qubits))
        F *= (1 - err[e]) ** (3 if i.operation.name == "swap" else 1)
    return F

# ---------------- Step 1: VF2Layout on the original circuit ----------------------
pm = PassManager(VF2Layout(target=target)); pm.run(qc)
print("\n[1] VF2Layout:", pm.property_set["VF2Layout_stop_reason"].name)

# ---------------- Step 2: SabreLayout (layout + SWAP) ----------------------------
sabre = PassManager(SabreLayout(cm, seed=2)).run(qc)
print("\n[2] After SabreLayout")
print(sabre.draw())
print(f"swaps = {sabre.count_ops().get('swap', 0)} | F = {fidelity(sabre):.3f}")

# ---------------- Step 3: VF2PostLayout on the routed circuit --------------------
pm = PassManager([SabreLayout(cm, seed=2),
                  VF2PostLayout(target=target, seed=2, strict_direction=False),
                  ApplyLayout()])
post = pm.run(qc)
print("\n[3] VF2PostLayout:", pm.property_set["VF2PostLayout_stop_reason"].name)
print(post.draw())
print(f"swaps = {post.count_ops().get('swap', 0)} | F = {fidelity(post):.3f}")

                    
q_0: ──■─────────■──
     ┌─┴─┐       │  
q_1: ┤ X ├──■────┼──
     └───┘┌─┴─┐┌─┴─┐
q_2: ─────┤ X ├┤ X ├
          └───┘└───┘

[1] VF2Layout: NO_SOLUTION_FOUND

[2] After SabreLayout
              ┌───┐   ┌───┐
q_2 -> 1 ─────┤ X ├───┤ X ├
         ┌───┐└─┬─┘   └─┬─┘
q_1 -> 2 ┤ X ├──■───X───■──
         └─┬─┘      │      
q_0 -> 3 ──■────────X──────
                           
swaps = 1 | F = 0.885

[3] VF2PostLayout: SOLUTION_FOUND
                           
q_0 -> 2 ──■────────X──────
         ┌─┴─┐      │      
q_1 -> 3 ┤ X ├──■───X───■──
         └───┘┌─┴─┐   ┌─┴─┐
q_2 -> 4 ─────┤ X ├───┤ X ├
              └───┘   └───┘
swaps = 1 | F = 0.941


# Exercise 4.1

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator, Clifford
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import UnitarySynthesis, HighLevelSynthesis, BasisTranslator
from qiskit.circuit.equivalence_library import SessionEquivalenceLibrary as sel

basis = ["cx", "rz", "sx", "x"]

# 1-qubit unitary (T·H)
o = QuantumCircuit(1); o.h(0); o.t(0)
U1 = UnitaryGate(Operator(o), label="U_1q")

# 2-qubit unitary (CX · Rz · CX)
z = QuantumCircuit(2); z.cx(0, 1); z.rz(0.5, 1); z.cx(0, 1)
U2 = UnitaryGate(Operator(z), label="U_zz")

# 2-qubit Clifford (H, CX, S)
c = QuantumCircuit(2); c.h(0); c.cx(0, 1); c.s(1)
cliff = Clifford(c)

qc = QuantumCircuit(2)
qc.append(U1, [0])
qc.append(U2, [0, 1])
qc.append(cliff, [0, 1])

def show(title, circ):
    print(f"\n{'=' * 20} {title} {'=' * 20}")
    print(circ.draw(output="text", fold=-1))
    print("ops:", dict(circ.count_ops()))

show("INPUT", qc)
s1 = PassManager(UnitarySynthesis(basis_gates=basis)).run(qc)
show("AFTER UnitarySynthesis", s1)
s2 = PassManager(HighLevelSynthesis(basis_gates=basis, equivalence_library=sel)).run(s1)
show("AFTER HighLevelSynthesis", s2)
s3 = PassManager(BasisTranslator(sel, basis)).run(s2)
show("AFTER BasisTranslator", s3)
print("\nequivalent to input:", Operator(qc).equiv(Operator(s3)))


==================== INPUT ====================
     ┌──────┐┌───────┐┌───────────┐
q_0: ┤ U_1q ├┤0      ├┤0          ├
     └──────┘│  U_zz ││  Clifford │
q_1: ────────┤1      ├┤1          ├
             └───────┘└───────────┘
ops: {'unitary': 2, 'clifford': 1}

==================== AFTER UnitarySynthesis ====================
global phase: 11π/8
     ┌─────────┐   ┌────┐   ┌──────────┐┌────────┐┌───┐        ┌───┐         ┌────────┐ ┌───────────┐
q_0: ┤ Rz(π/2) ├───┤ √X ├───┤ Rz(3π/4) ├┤ Rz(-π) ├┤ X ├──■─────┤ X ├─────■───┤ Rz(-π) ├─┤0          ├
     └──┬───┬──┘┌──┴────┴──┐└──────────┘└────────┘└───┘┌─┴─┐┌──┴───┴──┐┌─┴─┐┌┴────────┴┐│  Clifford │
q_1: ───┤ X ├───┤ Rz(-π/2) ├───────────────────────────┤ X ├┤ Rz(0.5) ├┤ X ├┤ Rz(-π/2) ├┤1          ├
        └───┘   └──────────┘                           └───┘└─────────┘└───┘└──────────┘└───────────┘
ops: {'rz': 7, 'x': 3, 'cx': 2, 'sx': 1, 'clifford': 1}

==================== AFTER HighLevelSynthesis ====================
global phase: 11